# 10 — Tools & Agents

So far the model only produced *text*. **Tools** let it *act* — call your Python functions
to fetch live data, do exact math, query a database, or search your documents. **Agents**
go one step further: given a goal and a set of tools, the model decides *which* tools to
call, in *what order*, and keeps going until it can answer — all on its own.

This combination is what people mean by "AI agents." In this notebook:

1. Defining a **tool** with the `@tool` decorator.
2. **Tool calling** by hand, so you understand the mechanics.
3. Building an agent with **`create_agent`** (LangChain 1.0, powered by LangGraph).
4. A **multi-tool** agent that picks the right tool per step.
5. Turning our **RAG retriever into a tool** (agentic RAG).
6. Giving an agent **memory** across turns.

> A note on scope: agents are powerful but add cost and unpredictability (the model might
> call tools you didn't expect). For a fixed pipeline, a plain LCEL chain (notebook 05) is
> cheaper and more reliable. Reach for an agent when the *sequence of steps isn't known in
> advance*.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("Model ready.")

## 1. Defining tools

A tool is just a Python function wrapped with `@tool`. LangChain turns it into a schema the
model can understand:

- the **function name** becomes the tool name,
- the **docstring** becomes the description the model reads to decide *when* to use it,
- the **type hints** define the arguments.

So your docstring and type hints aren't optional niceties here — they are the model's
instructions. Write them clearly.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    '''Multiply two numbers together.'''
    return a * b

@tool
def get_word_length(word: str) -> int:
    '''Return the number of letters in a single word.'''
    return len(word)

# Inspect what the model will see:
print("name       :", multiply.name)
print("description:", multiply.description)
print("args       :", multiply.args)

# A tool can also be called directly, like a function:
print("\nmultiply(6, 7) =", multiply.invoke({"a": 6, "b": 7}))

## 2. Tool calling, by hand

Before using an agent, let's see the raw mechanic it automates. You attach tools to a model
with `.bind_tools(...)`. Now when you invoke it, the model may respond not with text but
with a **request to call a tool** — a `tool_calls` list saying which tool and which
arguments.

The model does *not* run your function. It only asks. You execute the call, then send the
result back so the model can finish.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tools = [multiply, get_word_length]
model_with_tools = model.bind_tools(tools)

question = "What is 23.5 multiplied by 8?"
ai_msg = model_with_tools.invoke([HumanMessage(content=question)])

print("Text content:", repr(ai_msg.content))     # often empty when calling a tool
print("Tool calls  :", ai_msg.tool_calls)         # the model's request

In [ ]:
# We execute the requested call(s) and wrap each result in a ToolMessage.
tool_map = {t.name: t for t in tools}

tool_messages = []
for call in ai_msg.tool_calls:
    result = tool_map[call["name"]].invoke(call["args"])
    tool_messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    print(f"ran {call['name']}{call['args']} -> {result}")

# Send the conversation back (question + the model's request + our results) for the final answer.
final = model_with_tools.invoke([HumanMessage(content=question), ai_msg, *tool_messages])
print("\nFinal answer:", final.content)

That request → execute → respond loop is the whole game. For one tool call it's
manageable; but when a task needs several calls in sequence (and the model must decide each
step), writing the loop by hand gets messy. That's what an agent does for you.

## 3. `create_agent` — the automatic loop

`create_agent(model, tools, system_prompt=...)` builds an agent that runs the loop until
the task is done. Under the hood it's a small **LangGraph** state machine, but you don't
need to know graph internals to use it.

You invoke it with a `{"messages": [...]}` dict and get back the full message list. The
final answer is the last message.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools=[multiply, get_word_length],
    system_prompt="You are a helpful assistant. Use the tools when they apply.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "How many letters are in 'banana', and what is that number times 10?"}
]})

# The agent had to call get_word_length, then multiply. Show the whole trace:
for m in result["messages"]:
    m.pretty_print()

Read the trace top to bottom: the user's question, the model deciding to call
`get_word_length`, the tool result, the model calling `multiply`, that result, and finally
the natural-language answer. The agent sequenced two tools without us writing any loop.

In application code you usually just want the final text:

In [ ]:
print(result["messages"][-1].content)

## 4. A multi-tool agent

Give the agent several tools and it will pick whichever fits each step. Here we add a
(fake) weather tool. Notice the model uses the math tools or the weather tool depending on
the question — and ignores tools it doesn't need.

In [ ]:
@tool
def get_temperature(city: str) -> str:
    '''Get the current temperature for a city. Returns a short description.'''
    # A real tool would call a weather API; we fake it for the demo.
    fake = {"lahore": "38C and sunny", "oslo": "9C and cloudy", "cairo": "34C and clear"}
    return fake.get(city.lower(), "data unavailable for that city")

multi_agent = create_agent(
    model,
    tools=[multiply, get_word_length, get_temperature],
    system_prompt="You are a concise assistant. Use tools when helpful.",
)

for q in [
    "What's the weather in Oslo right now?",
    "What is 144 divided by... actually, what is 12 times 12?",
]:
    out = multi_agent.invoke({"messages": [{"role": "user", "content": q}]})
    print("Q:", q)
    print("A:", out["messages"][-1].content, "\n")

## 5. Agentic RAG: the retriever as a tool

In notebook 09, RAG *always* retrieved. But sometimes the user just says "hello" — no
search needed. If we wrap the retriever in a **tool**, the agent decides *whether* to search
the knowledge base. This is "agentic RAG": retrieval on demand.

First rebuild the retriever (load → split → embed → store), then expose it as a tool.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

assert os.path.exists("data/lumina_handbook.md"), "Run `python make_data.py` first."

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
docs = DirectoryLoader("data", glob="**/*.md", loader_cls=TextLoader,
                       loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(docs)
retriever = InMemoryVectorStore.from_documents(chunks, embedding=embeddings).as_retriever(search_kwargs={"k": 3})

@tool
def search_lumina_docs(query: str) -> str:
    '''Search the Lumina Robotics knowledge base for facts about the Pixel robot:
    pricing, battery, warranty, returns, support hours, languages, and specifications.
    Use this whenever the user asks about Lumina or Pixel.'''
    found = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in found)

print("Retriever tool ready.")

In [ ]:
support_agent = create_agent(
    model,
    tools=[search_lumina_docs],
    system_prompt=(
        "You are Lumina Robotics support. For any question about Lumina or the Pixel robot, "
        "use the search_lumina_docs tool and answer from what it returns. "
        "If the tool has no relevant info, say you don't know. For small talk, just reply."
    ),
)

# A small-talk message: the agent should NOT search.
print(support_agent.invoke({"messages": [{"role": "user", "content": "Hi there!"}]})["messages"][-1].content)
print("---")
# A factual message: the agent SHOULD search the docs.
print(support_agent.invoke({"messages": [{"role": "user", "content": "How much does the Pixel Pro cost and what's the warranty?"}]})["messages"][-1].content)

The agent skipped the tool for "Hi there!" and used it for the product question —
deciding for itself when retrieval was warranted. That judgment is the difference between
an agent and a fixed chain.

## 6. Giving the agent memory

By default each `invoke` is independent. To make the agent remember across turns, pass a
**checkpointer**. It saves the conversation state per `thread_id`, so subsequent calls with
the same id continue the same conversation — the agent equivalent of notebook 06's session
history.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model,
    tools=[multiply, get_temperature],
    system_prompt="You are a helpful, friendly assistant.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "demo-thread"}}

memory_agent.invoke({"messages": [{"role": "user", "content": "My name is Sara and I'm in Cairo."}]}, config=config)
out = memory_agent.invoke({"messages": [{"role": "user", "content": "What's the temperature where I am?"}]}, config=config)
print(out["messages"][-1].content)

The agent remembered Sara is in Cairo (from the first turn) and called the weather
tool for the right city in the second — memory and tools working together. For production
you'd swap `InMemorySaver` for a persistent checkpointer (SQLite, Postgres).

> **Gotchas.**
> - **Docstrings are prompts.** If an agent calls the wrong tool or skips one, sharpen the
>   tool descriptions first.
> - **Keep tools focused and safe.** A tool that does one clear thing is easier for the
>   model to use correctly. Be careful exposing tools with side effects (writes, payments).
> - **Agents can loop.** If a tool keeps failing, the agent may retry repeatedly. LangGraph
>   enforces a recursion limit, but design tools to fail clearly with a useful message.

## Your turn

Five exercises that build agents you could actually deploy — a bill calculator, an inventory
checker, a multi-tool store assistant, retrieval-on-demand, and a memory-plus-tool travel
assistant. Try each in a blank cell before opening its solution.

**Exercise 1 — Bill calculator agent.** Write an `order_total(subtotal, tax_percent,
tip_percent)` tool and an agent that uses it to answer a natural-language bill question
(exact math the model shouldn't do in its head).

*Sample run (illustrative):*

```text
Your total comes to $68.04 — that's $54.00 plus 8% tax and an 18% tip.
```

<details><summary>Show solution</summary>

```python
@tool
def order_total(subtotal: float, tax_percent: float, tip_percent: float) -> float:
    '''Compute an order total from a subtotal plus tax and tip percentages.'''
    return round(subtotal * (1 + tax_percent / 100 + tip_percent / 100), 2)

agent = create_agent(model, tools=[order_total],
                     system_prompt="Use the tool for any bill math.")
print(agent.invoke({"messages": [{"role": "user",
    "content": "My bill is $54, with 8% tax and an 18% tip. What's the total?"}]})["messages"][-1].content)
```

Offloading arithmetic to a tool is far more reliable than trusting the model to compute it.

</details>

**Exercise 2 — Inventory checker.** Write a `check_stock(item)` tool backed by a small dict and
build a store agent. Ask about two items at once and watch the agent call the tool for each —
the docstring is what tells it when to use it.

*Sample run (illustrative):*

```text
The blue shirt is in stock — we have 12 available. Unfortunately the red hat is currently
out of stock.
```

<details><summary>Show solution</summary>

```python
@tool
def check_stock(item: str) -> str:
    '''Check whether a clothing item is in stock. Input is a product name like "blue shirt".'''
    inventory = {"blue shirt": 12, "red hat": 0, "green socks": 5}
    n = inventory.get(item.lower())
    if n is None:
        return f"'{item}' is not in our catalog."
    return f"{item}: {n} in stock" if n else f"{item}: out of stock"

shop_agent = create_agent(model, tools=[check_stock],
    system_prompt="You are a store assistant. Use check_stock for availability questions.")
print(shop_agent.invoke({"messages": [{"role": "user",
    "content": "Do you have the blue shirt and the red hat?"}]})["messages"][-1].content)
```

The agent calls the tool once per item, then combines the results into one reply.

</details>

**Exercise 3 — Multi-tool store assistant.** Combine `order_total` and `check_stock` in one
agent, then send a stock question and a bill question and confirm it routes each to the right
tool.

*Sample run (illustrative):*

```text
Q: Are the green socks in stock?
A: Yes — the green socks are in stock, with 5 available.

Q: What's the total on a $30 order with 5% tax and no tip?
A: The total is $31.50.
```

<details><summary>Show solution</summary>

```python
combo = create_agent(model, tools=[order_total, check_stock],
    system_prompt="You are a store assistant. Use the right tool for each request.")

for q in ["Are the green socks in stock?",
          "What's the total on a $30 order with 5% tax and no tip?"]:
    print("Q:", q)
    print("A:", combo.invoke({"messages": [{"role": "user", "content": q}]})["messages"][-1].content, "\n")
```

The model reads each tool's description and picks the one that fits — no routing code from you.

</details>

**Exercise 4 — Retrieval on demand.** Send the `support_agent` (from section 5) a plain
thank-you and a genuine product question. Confirm it skips the search tool for small talk but
uses it — visible in the trace — for the product question.

*Sample run (illustrative — trace trimmed):*

```text
small talk: You're welcome — have a great day!

trace for the product question:
================================ Human Message =================================
What languages does Pixel Pro support?
================================== Ai Message ==================================
Tool Calls:
  search_lumina_docs(query='Pixel Pro supported languages')
================================= Tool Message =================================
Pixel Pro supports English, Spanish, French, German, and Japanese.
================================== Ai Message ==================================
Pixel Pro supports five languages: English, Spanish, French, German, and Japanese.
```

<details><summary>Show solution</summary>

```python
print("small talk:", support_agent.invoke({"messages": [
    {"role": "user", "content": "Thanks, have a good day!"}]})["messages"][-1].content)

res = support_agent.invoke({"messages": [
    {"role": "user", "content": "What languages does Pixel Pro support?"}]})
print("\ntrace for the product question:")
for m in res["messages"]:
    m.pretty_print()
```

Deciding *whether* to retrieve is exactly what separates agentic RAG from a fixed chain.

</details>

**Exercise 5 — Memory + tool travel assistant.** Give an agent a `get_temperature` tool and a
checkpointer. Tell it your destination in one turn, then ask "what's the temperature where I'm
headed?" and confirm it remembers the city and calls the tool for it.

*Sample run (illustrative):*

```text
It's currently 21C in Tokyo — pack some light layers for your trip tomorrow!
```

<details><summary>Show solution</summary>

```python
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_temperature(city: str) -> str:
    '''Get the current temperature for a city.'''
    fake = {"lahore": "38C", "oslo": "9C", "cairo": "34C", "tokyo": "21C"}
    return fake.get(city.lower(), "data unavailable")

assistant = create_agent(model, tools=[get_temperature],
    system_prompt="You are a friendly travel assistant.",
    checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "trip-1"}}
assistant.invoke({"messages": [{"role": "user", "content": "I'm flying to Tokyo tomorrow."}]}, config=cfg)
out = assistant.invoke({"messages": [{"role": "user", "content": "What's the temperature where I'm headed?"}]}, config=cfg)
print(out["messages"][-1].content)
```

Memory supplies "Tokyo" from the first turn; the tool supplies the temperature — together they
answer a question neither could alone.

</details>

## Summary

- A **tool** is a `@tool`-decorated function; its name, docstring, and type hints tell the
  model what it does and how to call it.
- `.bind_tools()` lets a model *request* tool calls (a `tool_calls` list); you execute them
  and return `ToolMessage`s — the loop an agent automates.
- **`create_agent(model, tools, system_prompt=...)`** builds a LangGraph agent that runs
  that loop until done; invoke with `{"messages": [...]}` and read the last message.
- Agents pick among **multiple tools**, and a retriever-as-a-tool gives you **agentic RAG**
  (search only when needed).
- Add a **checkpointer** + `thread_id` for cross-turn memory.

**Next:** [11 — Capstone Project](11_Capstone_Project.ipynb), where we combine RAG, tools,
structured output, and memory into one complete assistant.